## LIBRARIES AND DECLARATION

In [1]:
import pandas as pd
import json
import re
from IPython.display import FileLink

## CONFIG

In [2]:
STAGE = 1   # Stage 1: 3 attempts | Stage 2: 2 attempts

CONFIG = {
    1: {
        "files": [
            "/kaggle/input/private-dataset/step2_logic1_qwen3.7-max_1.csv",
            "/kaggle/input/private-dataset/step2_logic1_qwen3.7-max_2.csv",
            "/kaggle/input/private-dataset/step2_logic1_qwen3.7-max_3.csv",
        ],
        "out": "step1_classify1_logic1.csv",
    },
    2: {
        "files": [
            "/kaggle/input/datasets/dhuyent/step2-claude-gpt-2rd-stage/step1_filter1_logic1_claude-opus-4-8_1.csv",
            "/kaggle/input/datasets/dhuyent/step2-claude-gpt-2rd-stage/step1_filter1_logic1_claude-opus-4-8_2.csv",
        ],
        "out": "step2_classify1_logic1.csv",
    },
}

FILES       = CONFIG[STAGE]["files"]
OUT_PATH    = CONFIG[STAGE]["out"]
N_ATTEMPTS  = len(FILES)          # số lần chạy = số file
print(f"STAGE {STAGE}: {N_ATTEMPTS} attempts -> {OUT_PATH}")

STAGE 1: 3 attempts -> step1_classify1_logic1.csv


## LOAD DATA

In [3]:
df = pd.concat([pd.read_csv(f) for f in FILES], ignore_index=False)
print("Number of ids:", df.shape[0])
print("Unique ids:", df["id"].nunique())
df.head()

Number of ids: 537
Unique ids: 179


,topic,level,id,problem,solution,url,buggy_submission,bug_description,gt_status,gt_input,...,gt_reason,llm_model,prompt_strategy,pred_status,pred_input,pred_actual_output,pred_expected_output,pred_reason,comp.,label
0,String,Easy,1,Write a function to find the longest common pr...,class Solution {\n public:\n string longestC...,https://leetcode.com/problems/longest-common-p...,class Solution {\n public:\n string longestC...,[Logic – Wrong condition operator] Changed the...,Wrong Answer,"strs =\n[""flower"",""flow"",""flight""]",...,NaN,qwen3.7-max,zero-shot,Wrong Answer,"""[\""flower\"",\""flow\"",\""flight\""]""","""\""\""""","""\""fl\""""",The condition in the if-statement checks for c...,sim.,NaN
1,String,Easy,2,Given a string s containing just the character...,#include <iostream>\n#include <string>\n#inclu...,https://leetcode.com/problems/valid-parentheses,#include <iostream>\n#include <string>\n#inclu...,[Logic – Wrong condition operator] Changed the...,Wrong Answer,"s =\n""()""",...,NaN,qwen3.7-max,zero-shot,Wrong Answer,"""()""","""false""","""true""",NaN,sim.,NaN
2,String,Easy,3,Given a string s consisting of words and space...,"#pragma GCC target(""abm"")\n#pragma GCC target(...",https://leetcode.com/problems/length-of-last-word,"#pragma GCC target(""abm"")\n#pragma GCC target(...",[Logic – Wrong condition operator] Changed the...,Wrong Answer,"s =\n""Hello World""",...,NaN,qwen3.7-max,zero-shot,Wrong Answer,"""Hello World""","""0""","""5""",The second while loop incorrectly checks for s...,sim.,NaN
3,String,Easy,4,"Given a string s, return the number of segment...","#pragma GCC optimize(""Ofast"")\n\n#include <bit...",https://leetcode.com/problems/number-of-segmen...,"#pragma GCC optimize(""Ofast"")\n\n#include <bit...",[Logic – Wrong arithmetic operator] Changed th...,Wrong Answer,"s =\n""Hello, my name is John""",...,NaN,qwen3.7-max,zero-shot,Wrong Answer,"""Hello, my name is John""","""-5""","""5""",NaN,sim.,NaN
4,String,Easy,5,You are given a license key represented as a s...,"#pragma GCC optimize(""Ofast"")\n\n#include <bit...",https://leetcode.com/problems/license-key-form...,"#pragma GCC optimize(""Ofast"")\n\n#include <bit...",[Logic – Wrong condition operator] Changed the...,Wrong Answer,"s =\n""5F3Z-2e-9-w""\nk =\n4",...,NaN,qwen3.7-max,zero-shot,Wrong Answer,"""s = \""5F3Z-2e-9-w\"", k = 4""","""\""5F3-Z2E9-W\""""","""\""5F3Z-2E9W\""""",The code incorrectly counts the number of dash...,sim.,NaN


## LABEL CLASSIFICATION

Classify each `id` based on how well `pred_status` (LLM prediction) matches `gt_status` (ground truth) across multiple runs (attempts).

- **Label 1**: ALL attempts match (`pred_status == gt_status`)
- **Label 2**: SOME but not all attempts match (partial match)
- **Label 3**: NO attempt matches

In [4]:
# So sánh gt_status vs pred_status
df["match"] = df.apply(
    lambda r: "sim" if r["gt_status"] == r["pred_status"] else "diff",
    axis=1
)

# Group theo id, đếm sim - diff trong mỗi nhóm
group_stats = (
    df.groupby("id")["match"]
    .value_counts()
    .unstack(fill_value=0)
    .rename(columns={"sim": "n_sim", "diff": "n_diff"})
    .reset_index()
)

# Đảm bảo cả hai cột luôn tồn tại dù tất cả là sim hoặc diff
for col in ("n_sim", "n_diff"):
    if col not in group_stats.columns:
        group_stats[col] = 0

In [5]:
# Check if ids whose `pred_status` differs on EVERY attempt
uniq = df.groupby("id")["pred_status"].nunique().reset_index(name="n_unique")
cnt  = df.groupby("id").size().reset_index(name="n_rows")
uniq = uniq.merge(cnt, on="id")

ids_all_diff = uniq.loc[
    (uniq["n_rows"] == N_ATTEMPTS) & (uniq["n_unique"] == N_ATTEMPTS),
    "id",
]

print("Number of ids with a different pred_status on every attempt:", len(ids_all_diff))
print(ids_all_diff.tolist())

Number of ids with a different pred_status on every attempt: 0
[]


In [6]:
def assign_label(row, n_attempts=N_ATTEMPTS):
    s, d = row["n_sim"], row["n_diff"]
    total = s + d
    if total != n_attempts:
        return None          # group has missing/extra rows: flag for inspection
    if s == total:           # all sim
        return 1
    if d == total:           # all diff
        return 3
    return 2                 # partial match

group_stats["label"] = group_stats.apply(assign_label, axis=1)

df = df.drop(columns=["label"], errors="ignore")
df = df.merge(group_stats[["id", "label"]], on="id", how="left")

print(df[["id", "gt_status", "pred_status", "match", "label"]].head(20))
print()
print("Label distribution (per id):")
print(df.drop_duplicates("id")["label"].value_counts(dropna=False).sort_index())

    id     gt_status   pred_status match  label
0    1  Wrong Answer  Wrong Answer   sim      1
1    2  Wrong Answer  Wrong Answer   sim      1
2    3  Wrong Answer  Wrong Answer   sim      1
3    4  Wrong Answer  Wrong Answer   sim      1
4    5  Wrong Answer  Wrong Answer   sim      1
5    6  Wrong Answer  Wrong Answer   sim      1
6    7  Wrong Answer  Wrong Answer   sim      1
7    8  Wrong Answer  Wrong Answer   sim      1
8    9  Wrong Answer  Wrong Answer   sim      1
9   10  Wrong Answer  Wrong Answer   sim      1
10  11  Wrong Answer  Wrong Answer   sim      1
11  12  Wrong Answer  Wrong Answer   sim      1
12  13  Wrong Answer  Wrong Answer   sim      1
13  14  Wrong Answer  Wrong Answer   sim      1
14  15  Wrong Answer  Wrong Answer   sim      1
15  16  Wrong Answer  Wrong Answer   sim      1
16  17  Wrong Answer  Wrong Answer   sim      1
17  18  Wrong Answer  Wrong Answer   sim      1
18  19  Wrong Answer  Wrong Answer   sim      1
19  20  Wrong Answer  Wrong Answer   sim

## SAVE OUTPUT

In [7]:
df_out = df.drop_duplicates("id").reset_index(drop=True)
df_out.to_csv(OUT_PATH, index=False)
print("Saved", df_out.shape[0], "ids to", OUT_PATH)

Saved 179 ids to step1_classify1_logic1.csv


In [8]:
FileLink(OUT_PATH)

/kaggle/working/step1_classify1_logic1.csv